# Qwen3-4B on Spider dev: RAG and ReAct ablation

| variant | few-shot RAG (5 nearest train examples) | ReAct loop | act tools | Report Name |
|---|---|---|---|---|
| `base` | – | – | – (the original A1 prompt) | System A |
| `rag` | ✓ | – | – | System A + Few-Shot RAG |
| `react` | – | ✓ | `execute` | System A + ReAct Loop + Execute Tool | 
| `react_parse` | – | ✓ | `execute` + `spider_parse` | System A + ReAct Loop + Spider Parse Tool |
| `rag_react` | ✓ | ✓ | `execute` | System A + Few-Shot RAG + ReAct Loop + Execute Tool |
| `rag_react_parse` | ✓ | ✓ | `execute` + `spider_parse` | System B (Few-Shot RAG + ReAct Loop + Execute Tool + Spider Parse Tool)|


In [ ]:
import contextlib, io, os, sys
from IPython.display import Markdown, display

# run from the repo root; importing agent_ext puts a1_lib and spider on sys.path and sets NLTK_DATA
sys.path.insert(0, os.getcwd())
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
from agent_ext import REPO, RESULTS_DIR
from agent_ext import run as ext_run, compare as ext_compare

RUN_ID = "repro"      # every variant writes to agent_ext/results/<variant>/<RUN_ID>/
LIMIT = None          # None = full dev (1034); a small number (e.g. 20) for a quick check
N_TRACES = 5          # random traces saved per variant, from successes and from failures each
SEED = 0
BATCH_SIZE = 64       # questions generated together; halved automatically on CUDA out-of-memory
VARIANTS = list(ext_run.VARIANTS)   # base, rag, react, react_parse, rag_react, rag_react_parse

print("repo:", REPO)

In [ ]:
def variant_args(variant, extra=()):
    a = ["--variant", variant, "--run-id", RUN_ID, "--seed", str(SEED),
         "--save-traces", str(N_TRACES), "--batch-size", str(BATCH_SIZE), "--local-only", "--models-path", "a1_lib/models", *extra]
    return a + (["--limit", str(LIMIT)] if LIMIT else [])

def run_main(main, args):
    """Call a script's main() in this process with the given command-line args."""
    old = sys.argv
    sys.argv = ["agent_ext", *args]
    try:
        main()
    finally:
        sys.argv = old

# run every variant (needs a GPU); rerunning a RUN_ID overwrites it, add "--resume" via `extra` to continue
os.chdir(REPO)
for variant in VARIANTS:
    print(f"=== {variant}", flush=True)
    run_main(ext_run.main, variant_args(variant))

In [ ]:
# paired McNemar tests and habit diagnostics between variants
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    run_main(ext_compare.main, ["--run-id", RUN_ID])
display(Markdown(buf.getvalue()))

In [ ]:
from agent_ext import report_tables as rt
MODELS_PATH = REPO / "a1_lib/models"

# RQ1: exact match, validity, component matching, whole-query metrics, agreement
for title, df in [("Exact match (%) by difficulty", rt.accuracy(RUN_ID)),
                  ("Output validity", rt.validity(RUN_ID)),
                  ("Component matching (%)", rt.components(RUN_ID)),
                  ("Whole-query metrics (execution is on schema-only databases)", rt.whole_query(RUN_ID)),
                  ("Agreement on exact match", rt.agreement(RUN_ID))]:
    display(Markdown(f"**{title}**"))
    display(df)

# RQ1: efficiency (tokens and wall-clock)
display(rt.efficiency(RUN_ID, MODELS_PATH))

# RQ2: ablation, repair success rate, semantic distance between first and final draft
display(rt.ablation(RUN_ID))
display(rt.repair_success(RUN_ID))
display(rt.semantic_distance(RUN_ID, MODELS_PATH))

In [ ]:
from IPython.display import Image
from agent_ext import plot_trajectories, final_exec

run_main(plot_trajectories.main, ["--run-id", RUN_ID])
run_main(final_exec.main, ["--run-id", RUN_ID])
FIGS = RESULTS_DIR / "figs"
for name in ["trajectories.png", "final_exec_errors.png"]:
    display(Image(filename=str(FIGS / name)))